# 🚀 AI-LAB Real GPU Acceleration Bridge
Welcome! This notebook provides a **free NVIDIA Tesla T4 GPU** backend for your Smart Canvas Grow AI-LAB.

### Instructions:
1. Ensure GPU is enabled: **Runtime** > **Change runtime type** > **T4 GPU**
2. Click the **▶ Play button** on the code cell below.
3. Copy the generated **Tunnel URL** and paste it into the website.

In [ ]:
# ==========================================================
# ⚡ Instant Real GPU Backend Server
# ==========================================================
!pip install flask flask-cors pyngrok --quiet

import sys, io, torch, threading, time
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok

app = Flask(__name__)
CORS(app)

@app.route("/api/status", methods=["GET"])
def status():
    has_cuda = torch.cuda.is_available()
    dev = torch.cuda.get_device_name(0) if has_cuda else "CPU"
    return jsonify({"status": "online", "device": dev, "cuda": has_cuda})

@app.route("/api/execute", methods=["POST"])
def execute():
    code = request.get_json(force=True).get("code", "")
    old_out, old_err = sys.stdout, sys.stderr
    sys.stdout = b_out = io.StringIO()
    sys.stderr = b_err = io.StringIO()
    try:
        exec(code, globals())
        out = b_out.getvalue() or b_err.getvalue() or "Executed successfully."
    except Exception as e:
        out = f"Execution Error: {str(e)}"
    finally:
        sys.stdout, sys.stderr = old_out, old_err
    return jsonify({
        "output": out,
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    })

# Start Flask server
threading.Thread(target=lambda: app.run(port=5000), daemon=True).start()
time.sleep(1)

# Generate live tunnel URL
public_url = ngrok.connect(5000).public_url
print("\n" + "="*55)
print("🚀 YOUR REAL GPU TUNNEL URL IS READY:")
print(f"👉 {public_url}")
print("="*55 + "\n")